# Brief 02 : SCD1, SCD2, SCD3, et la question du grain

Dans le brief 01 (remanié) , on a construit une table `produit_historise` : 
- une ligne par version d'un produit, avec `valid_from`, `valid_to` et `is_current`
- c'est un **SCD2**

Ce n'est pas la seule façon d'historiser une dimension. 

Il existe trois façons usuelles d'historiser :

| Type | Ce qu'on garde | Nombre de lignes par produit |
|---|---|---|
| **SCD1** | seulement l'état courant, on écrase | 1 |
| **SCD2** | toutes les versions, avec bornes de validité | autant que de versions |
| **SCD3** | l'état courant, plus la **valeur précédente** dans une colonne | 1 |

Le but de ce brief est d'étudier ces trois façons sur les mêmes données, ainsi que de mesurer ce que chacune coûte, et d'étudier les limites de chacune des méthodes.

### Un même produit, trois représentations

Point de départ : les observations d'un produit dont le prix passe de 23.99 à 19.99 entre deux relevés. C'est ce que contient déjà `produit_historise`, un SCD2.

```
produit_historise  (SCD2)
┌──────────┬───────────────┬───────┬────────────┬────────────┬────────────┐
│ site     │ cle           │ price │ valid_from │ valid_to   │ is_current │
├──────────┼───────────────┼───────┼────────────┼────────────┼────────────┤
│ vetoplus │ 3552791071358 │ 23.99 │ 2026-07-12 │ 2026-07-18 │ false      │
│ vetoplus │ 3552791071358 │ 19.99 │ 2026-07-18 │ (null)     │ true       │
└──────────┴───────────────┴───────┴────────────┴────────────┴────────────┘
```

Le même produit, historisé des trois façons :

```
SCD1  (état courant seul, on écrase)
┌──────────┬───────────────┬───────┐
│ site     │ cle           │ price │
├──────────┼───────────────┼───────┤
│ vetoplus │ 3552791071358 │ 19.99 │   <- l'ancien 23.99 est perdu
└──────────┴───────────────┴───────┘

SCD3  (état courant + valeur précédente)
┌──────────┬───────────────┬───────┬─────────────────┐
│ site     │ cle           │ price │ price_precedent │
├──────────┼───────────────┼───────┼─────────────────┤
│ vetoplus │ 3552791071358 │ 19.99 │ 23.99           │   <- une seule étape de mémoire
└──────────┴───────────────┴───────┴─────────────────┘

SCD2  (toutes les versions)  -> les 2 lignes de la table ci-dessus
```

💫 Résumé de l'arbitrage : 

- SCD1 est compact mais "per la mémoire"
- SCD3 garde un pas d'historique dans une colonne (i.e. deux versions consécutives : dernière et avant dernière)
- SCD2 garde tout au prix d'une ligne par version

> Prérequis : la table `produit_historise` du brief 01, chargée. 🟢 On ne relit aucun fichier de relevé :) !!

In [9]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*") # warning de SQLAlchemy
import pandas as pd
import psycopg2

# ⚠️ adaptez si besoin
conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
conn.autocommit = False
cur = conn.cursor()
cur = conn.cursor()

In [10]:
# La table du brief 01, chargée (aussi en pandas pour les réponses pandas)
ph = pd.read_sql("SELECT * FROM produit_historise", conn)
ph["price"] = pd.to_numeric(ph["price"])
print(ph.shape)
ph.head(3)

(331056, 11)


,site,cle,ean,url,name,brand,price,in_stock,valid_from,valid_to,is_current
0,animalis,0000000856126,0000000856126,https://www.animalis.com/animalis-friandises-a...,Animalis - Friandises au Fromage Souflé pour C...,Animalis,4.97,False,2026-07-12 12:42:29.292547+00:00,2026-07-18 13:55:43.109577+00:00,False
1,animalis,0000000856126,0000000856126,https://www.animalis.com/animalis-friandises-a...,Animalis - Friandises au Fromage Souflé pour C...,Animalis,4.97,False,2026-07-18 13:55:43.109577+00:00,NaT,True
2,animalis,0000003397510,0000003397510,https://www.animalis.com/foolee-brosse-pour-ch...,Foolee - Brosse pour Chien de Moyenne Race - M,Fluval,31.95,False,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00,False


### A1. État des lieux

Combien de versions contient `produit_historise` ? Combien de produits distincts `(site, cle)` ?

⚠️ C'est la variable la plus importante pour comparer les méthodes d'historisation.

In [11]:
conn.rollback()


In [35]:
# --- SQL ---
SCD2= pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM produit_historise) AS nb_versions_SCD2,
    (SELECT COUNT(DISTINCT (site, cle)) FROM produit_historise) AS nb_produits;
""", conn)
display(SCD2.head())




,nb_versions_scd2,nb_produits
0,331056,140947


In [31]:
# --- pandas ---

nb_versions_SCD2 = len(ph)
nb_produits = ph[["site", "cle"]].drop_duplicates().shape[0]

print("Nombre total de versions :", nb_versions_SCD2)
print("Nombre de produits distincts :", nb_produits)


Nombre total de versions : 331056
Nombre de produits distincts : 140947


### A2. Construire le SCD1

Le SCD1 ne garde que l'état courant : **une ligne par produit**, sans historique.

Produisez-le à partir de `produit_historise`, avec au moins `site`, `cle`, `name`, `price`. Combien de lignes obtient-on ?

In [29]:
# --- SQL ---
print("SQL==>")
SCD1= pd.read_sql(""" 
 SELECT site, cle, name, price
    FROM produit_historise WHERE is_current
    ORDER BY site, cle

""",conn)
display(SCD1.head())
print("Nombre de lignes SCD1 :", len(SCD1))
# --- pandas ---
print("pandas==>")
# 1. Filtrer l'état courant
scd1 = ph[ph["is_current"] == True][["site", "cle", "name", "price"]]

# 2. Nombre de lignes
nb_scd1 = scd1.shape[0]

display(scd1.head())
print("Nombre de lignes SCD1 :", nb_scd1)

SQL==>


,site,cle,name,price
0,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97
1,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95
2,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59
3,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69
4,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99


Nombre de lignes SCD1 : 140947
pandas==>


,site,cle,name,price
1,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97
4,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95
6,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59
8,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69
10,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99


Nombre de lignes SCD1 : 140947


### A3. Construire le SCD3

Le SCD3 garde une ligne par produit, plus une colonne permettant de garder la **valeur précédente** (i.e. on garde mémoire de l'avant dernier état)

Créer une ligne par produit avec deux colonnes de prix : `price` (courant) et `price_precedent` (nul s'il n'y en a pas).

Indice : `lag(price) OVER (PARTITION BY site, cle ORDER BY valid_from)`, puis ne garder que la dernière version.

In [28]:
# --- SQL ---
print("SQL==>")
SCD3= pd.read_sql(""" 
SELECT
    site,
    cle,
    name,
    price AS price_courant,
    lag_price AS price_precedent
FROM (
    SELECT
        site,
        cle,
        name,
        price,
        LAG(price) OVER (
            PARTITION BY site, cle
            ORDER BY valid_from
        ) AS lag_price,
        is_current
    FROM produit_historise
) AS t
WHERE is_current = TRUE;
""",conn)
display(SCD3.head())
print("Nombre de lignes SCD3 :", SCD3.shape[0])

# --- pandas ---
print("Pandas (A3) ==>")

# 1. Trier pour que lag fonctionne correctement
ph_sorted = ph.sort_values(["site", "cle", "valid_from"])

# 2. Calcul du prix précédent
ph_sorted["price_precedent"] = (
    ph_sorted.groupby(["site", "cle"])["price"].shift(1)
)

# 3. Garder seulement la version actuelle
scd3 = ph_sorted[ph_sorted["is_current"] == True][
    ["site", "cle", "name", "price", "price_precedent"]
]

display(scd3.head())
print("Nombre de lignes SCD3 :", scd3.shape[0])


SQL==>


,site,cle,name,price_courant,price_precedent
0,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97,4.97
1,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95,31.95
2,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59,1.59
3,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69,11.69
4,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99,27.99


Nombre de lignes SCD3 : 140947
Pandas (A3) ==>


,site,cle,name,price,price_precedent
1,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97,4.97
4,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95,31.95
6,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59,1.59
8,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69,11.69
10,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99,27.99


Nombre de lignes SCD3 : 140947


### A4. Comparer les volumétries

Mettez côte à côte, sur les mêmes données, le nombre de lignes de chacune des trois approches. Commentez l'écart en une phrase.

In [38]:
conn.rollback()


In [ ]:
# --- SQL ---
scd2_sql = pd.read_sql("""
SELECT COUNT(*) AS scd2_nb
FROM produit_historise;
""", conn)

scd1_sql = pd.read_sql("""
SELECT COUNT(*) AS scd1_nb
FROM produit_historise
WHERE is_current = TRUE;
""", conn)


scd3_sql = pd.read_sql("""
SELECT COUNT(*) AS scd3_nb
FROM (
    SELECT
        site,
        cle,
        LAG(price) OVER (
            PARTITION BY site, cle
            ORDER BY valid_from
        ) AS price_precedent,
        is_current
    FROM produit_historise
) AS t
WHERE is_current = TRUE;
""", conn)
print("Nombre de ligne en SQL:")
display(scd1_sql)
display(scd2_sql)
display(scd3_sql)

Nombre de ligne en SQL:


,scd1_nb
0,140947


,scd2_nb
0,331056


,scd3_nb
0,140947


In [41]:
# ---pandas---

# SCD2
scd2_nb = ph.shape[0]

# SCD1
scd1_nb = ph[ph["is_current"] == True].shape[0]

# SCD3
ph_sorted = ph.sort_values(["site", "cle", "valid_from"])
ph_sorted["price_precedent"] = ph_sorted.groupby(["site", "cle"])["price"].shift(1)
scd3_nb = ph_sorted[ph_sorted["is_current"] == True].shape[0]

print("Nombre de lignes en pandas:")
print("SCD1 :", scd1_nb)
print("SCD2 :", scd2_nb)
print("SCD3 :", scd3_nb)


Nombre de lignes en pandas:
SCD1 : 140947
SCD2 : 331056
SCD3 : 140947


#### ---> SCD2 est beaucoup plus volumineux que SCD1 et SCD3 (331 056 lignes contre 140 947), car il stocke toutes les versions alors que SCD1 et SCD3 ne gardent qu’une seule ligne par produit.

### A5. La question du grain

- Notre SCD2 crée une version **par observation**, même quand le prix n'a pas bougé. 
- Donc, on stocke de l'information qui a l'air redondante, **mais qui ne l'est pas forcément !**. 
- En effet, dans le cas de prix, une version supplémentaire avec le même prix mais à une autre date, indique que le prix n'a pas bougé à une certaine date.
- L'autre grain possible ne crée une version **que lorsque le prix change**. Mais on ne sait pas (dans cet exemple) **quand le prix a changé**.

Prenons un produit relevé trois fois, dont le prix ne bouge qu'à la fin :

```
grain « par observation » (le nôtre)          grain « par changement »
┌───────┬────────────┐                         ┌───────┬────────────┐
│ price │ valid_from │                         │ price │ valid_from │
├───────┼────────────┤                         ├───────┼────────────┤
│ 21.90 │ 2026-04-01 │                         │ 21.90 │ 2026-04-01 │
│ 21.90 │ 2026-07-12 │  <- prix inchangé,      │ 24.90 │ 2026-07-18 │
│ 24.90 │ 2026-07-18 │     version « inutile » └───────┴────────────┘
└───────┴────────────┘                         2 versions : une nouvelle ligne
3 versions                                     seulement quand le prix change
```

- Comptez combien de versions il resterait avec ce grain par changement, et donnez l'écart avec la table actuelle (i.e. avec le nombre de versions dans la table actuelle "par observations")

💫 Un petit indice : une version est utile si son prix diffère de celui de la version précédente du même produit, ou si c'est la première. Dans les cas contraires, elle n'est pas utile.

In [46]:
# --- SQL ---
print("SQL--->")
display( pd.read_sql("""

SELECT COUNT(*) AS nb_versions_par_changement
FROM (
    SELECT
        site,
        cle,
        price,
        LAG(price) OVER (
            PARTITION BY site, cle
            ORDER BY valid_from
        ) AS price_precedent
    FROM produit_historise
) AS t
WHERE price_precedent IS NULL      -- première version
   OR price <> price_precedent;    -- changement de prix
""",conn))

SQL--->


,nb_versions_par_changement
0,211021


In [ ]:
# --- pandas ---

ph_sorted = ph.sort_values(["site", "cle", "valid_from"])

ph_sorted["price_precedent"] = (
    ph_sorted.groupby(["site", "cle"])["price"].shift(1)
)

versions_par_changement = ph_sorted[
    (ph_sorted["price_precedent"].isna()) |
    (ph_sorted["price"] != ph_sorted["price_precedent"])
].shape[0]

print("Versions par changement :", versions_par_changement)
print("Versions actuelles (SCD2) :", ph.shape[0])
print("Écart :", ph.shape[0] - versions_par_changement)


Versions par changement : 211023
Versions actuelles (SCD2) : 331056
Écart : 120033


### ---> Avec un grain “par changement”, on conserverait 211 023 versions au lieu de 331 056, soit 120 033  versions (soit 36.3 %) en moins, mais on perdrait l’information sur les dates où le prix est resté stable.

#